In [0]:
#In This notebook iam creating fact tables and dimension tables
#fact sales are: fact_sales and fact_returns

In [0]:
from pyspark.sql.functions import (
    col,year,month,quarter,dayofmonth,dayofweek)

In [0]:
# Customer Dimension

dim_customer = (
    spark.table("silver_customers")
    .select(
        "CustomerKey",
        "Prefix",
        "FirstName",
        "LastName",
        "BirthDate",
        "MaritalStatus",
        "Gender",
        "EmailAddress",
        "AnnualIncome",
        "TotalChildren",
        "EducationLevel",
        "Occupation",
        "HomeOwner"
    )
    .dropDuplicates(["CustomerKey"])
)



In [0]:
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_customer")

In [0]:
# Product Dimension
# We will combine Products → Subcategory → Category.

dim_product = (
    spark.table("silver_products")
    .join(
        spark.table("silver_product_subcategories"),
        "ProductSubcategoryKey",
        "left"
    )
    .join(
        spark.table("silver_product_categories"),
        "ProductCategoryKey",
        "left"
    )
)

In [0]:
# -- Select the important columns:
dim_product = dim_product.select(
    "ProductKey",
    "ProductName",
    "ProductSubcategoryKey",
    "SubcategoryName",
    "ProductCategoryKey",
    "CategoryName",
    "ProductColor",
    "ProductSize",
    "ProductPrice",
    "ProductCost",
    "ProductStyle",
    "ModelName"
)



In [0]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_product")

In [0]:
#  Territory Dimension

dim_territory = (
    spark.table("silver_territories")
    .select(
        "SalesTerritoryKey",
        "Region",
        "Country",
        "Continent"
    )
    .dropDuplicates(["SalesTerritoryKey"])
)

dim_territory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_territory")

In [0]:
# Date Dimension
# We'll generate useful calendar attributes from your existing calendar.

dim_date = (
    spark.table("silver_calendar")
    .select("Date")
    .dropDuplicates(["Date"])
    .withColumn("Year", year(col("Date")))
    .withColumn("Month", month(col("Date")))
    .withColumn("Quarter", quarter(col("Date")))
    .withColumn("Day", dayofmonth(col("Date")))
    .withColumn("DayOfWeek", dayofweek(col("Date")))
)



In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

In [0]:
from pyspark.sql.functions import col

fact_sales = (
    spark.table("silver_sales")
    .join(
        spark.table("dim_product")
        .select(
            "ProductKey",
            "ProductPrice",
            "ProductCost"
        ),
        "ProductKey",
        "left"
    )
    .select(
        "OrderNumber",
        "OrderLineItem",
        "OrderDate",
        "CustomerKey",
        "ProductKey",
        "TerritoryKey",
        "OrderQuantity",
        "ProductPrice",
        "ProductCost"
    )
    .withColumn(
        "Revenue",
        col("OrderQuantity") * col("ProductPrice")
    )
    .withColumn(
        "TotalCost",
        col("OrderQuantity") * col("ProductCost")
    )
    .withColumn(
        "Profit",
        col("Revenue") - col("TotalCost")
    )
)

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_sales")

In [0]:
# Fact Returns

fact_returns = (
    spark.table("silver_returns")
    .select(
        "ReturnDate",
        "TerritoryKey",
        "ProductKey",
        "ReturnQuantity"
    )
)


In [0]:

fact_returns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_returns")

In [0]:
#  Verifying  Star Schema

star_tables = [
    "dim_customer",
    "dim_product",
    "dim_territory",
    "dim_date",
    "fact_sales",
    "fact_returns"
]



In [0]:
for table in star_tables:
    print(
        f"{table}: {spark.table(table).count()} rows"
    )

dim_customer: 18148 rows
dim_product: 293 rows
dim_territory: 10 rows
dim_date: 912 rows
fact_sales: 56046 rows
fact_returns: 1809 rows


In [0]:
spark.table("fact_sales").printSchema()

root
 |-- OrderNumber: string (nullable = true)
 |-- OrderLineItem: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerKey: integer (nullable = true)
 |-- ProductKey: integer (nullable = true)
 |-- TerritoryKey: integer (nullable = true)
 |-- OrderQuantity: integer (nullable = true)
 |-- ProductPrice: double (nullable = true)
 |-- ProductCost: double (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- TotalCost: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
display(spark.table("fact_sales").limit(5))

OrderNumber,OrderLineItem,OrderDate,CustomerKey,ProductKey,TerritoryKey,OrderQuantity,ProductPrice,ProductCost,Revenue,TotalCost,Profit
SO45081,1,2015-01-01,26782,338,6,1,699.0982,413.1463,699.0982,413.1463,285.9519
SO45092,1,2015-01-03,18899,313,9,1,3578.27,2171.2942,3578.27,2171.2942,1406.9758000000002
SO45109,1,2015-01-07,14937,311,10,1,3578.27,2171.2942,3578.27,2171.2942,1406.9758000000002
SO45117,1,2015-01-08,14727,342,1,1,699.0982,413.1463,699.0982,413.1463,285.9519
SO45138,1,2015-01-11,11452,349,9,1,3374.99,1898.0944,3374.99,1898.0944,1476.8955999999998
